# Process GenX CEM Results

In [ ]:
from ipywidgets import Dropdown, Select, SelectMultiple, Text
from upath import UPath
from src import runner
from ipyfilechooser import FileChooser
import xlwings as xw
from loguru import logger
from tqdm.notebook import trange, tqdm
from datetime import date

In [ ]:
genx_wb = FileChooser(default_path=".", default_filename="Kentucky Load Resource Model.xlsb", title="Connect to a GenX spreadsheet: ", filter_pattern="*.xls*", show_hidden=False)
genx_wb

## Load single case results into active spreadsheet

This will load the case selected below into the active GenX spreadsheet for results reporting/charts.

In [ ]:
selected_case = Select(
    options=runner.get_solved_cases(UPath("./cases")),
    description="Available CEM cases: ",
    rows=10,
    layout=dict(width="max-content"),
    style=dict(description_width="max-content"),
)
selected_case

In [ ]:
logger.info(f"Opening {genx_wb.value} in new Excel instance")

with xw.App():
    results_wb = xw.Book(genx_wb.value)
    
    results_wb.app.screen_updating = False    
    results_wb.app.calculate()
    
    try:
        runner.load_case_results(
            wb=results_wb,
            base_folder=UPath(selected_case.value),
            save_view=False,
        )
    except Exception:
        logger.error(f"Oops! Skipping {case_folder}.")

    results_wb.app.screen_updating = True

## Compare scenario results

This will create a new spreadsheet that contains "snapshots" of the `GenX Results` tab from your active GenX spreadsheet for each scenario you select below. You can then use `INDIRECT` and other formulas to compare across scenario results.

In [ ]:
selected_cases = SelectMultiple(
    options=runner.get_solved_cases(UPath("./cases")),
    description="Available CEM cases: ",
    rows=10,
    layout=dict(width="max-content"),
    style=dict(description_width="max-content"),
)
selected_cases

In [ ]:
report_wb_name = Text(
    value=f'Compiled Results {date.today()}',
    # placeholder='Name the results comparison workbook',
    description='Name the results comparison workbook:',
    layout=dict(width="800px"),
    style=dict(description_width="max-content"),
)
report_wb_name

In [ ]:
logger.info(f"Opening {genx_wb.value} in new Excel instance")

with xw.App():
    results_wb = xw.Book(genx_wb.value)
    
    results_wb.app.screen_updating = False    
    results_wb.app.calculate()
    
    for case_folder in tqdm(selected_cases.value, desc="Processing results"):
        try:
            report_wb = runner.load_case_results(
                wb=results_wb,
                base_folder=UPath(case_folder),
                save_view=True,
                report_wb_template_path="./Compiled Results.xlsx",
            )
        except Exception:
            logger.error(f"Oops! Skipping {case_folder}.")
    
    try:
        report_wb.save(f"{report_wb_name.value}.xlsx")
        logger.success(f"Scenario comparison spreadsheet saved at: {report_wb_name.value}.xlsx")
    except Exception:
        logger.error("Oops! Seems like we couldn't save the scenario comparison spreadsheet. This might be because a file with the same name is already currently open in Excel.")

    results_wb.app.screen_updating = True